In [ ]:
import io
import json
import pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
# the data was copied from https://www.bls.gov/charts/american-time-use/activity-by-sex.htm

df_gt = pd.read_csv(io.StringIO('''
activity,hours_spent_gt
"Personal care, including sleep",9.8
Eating and drinking,1.24
Household activities,2.01
Purchasing goods and services,0.67
Caring for and helping household members,0.51
Caring for and helping nonhousehold members,0.17
Working and work-related activities,3.43
Educational activities,0.42
"Organizational, civic, and religious activities",0.3
Leisure and sports,5.07
"Telephone calls, mail, and e-mail",0.18
"Other activities, not elsewhere classified",0.21
'''))

df_gt

,activity,hours_spent_gt
0,"Personal care, including sleep",9.80
1,Eating and drinking,1.24
2,Household activities,2.01
3,Purchasing goods and services,0.67
4,Caring for and helping household members,0.51
5,Caring for and helping nonhousehold members,0.17
6,Working and work-related activities,3.43
7,Educational activities,0.42
8,"Organizational, civic, and religious activities",0.30
9,Leisure and sports,5.07


In [ ]:
#@title Technical notes

technical_notes = """
Technical Note


   The estimates in this news release are based on annual average data from the American Time Use Survey (ATUS).
The ATUS, which is conducted by the U.S. Census Bureau for the Bureau of Labor Statistics (BLS), is a
continuous survey about how individuals age 15 and over spend their time.

   If you are deaf, hard of hearing, or have a speech disability, please dial 7-1-1 to access telecommunications
relay services.

Survey methodology

   Data collection for the ATUS began in January 2003. Sample cases for the survey are selected monthly, and
interviews are conducted continuously throughout the year. In 2024, approximately 7,700 individuals were
interviewed. Estimates are released annually.

   ATUS sample households are chosen from the households that completed their eighth (final) interview for the
Current Population Survey (CPS), the nation’s monthly household labor force survey. ATUS sample households
are selected to ensure that estimates will be nationally representative.

   One individual age 15 or over is randomly chosen from each sampled household. This "designated person" is
interviewed by telephone once about his or her activities on the day before the interview--the "diary day."

   All ATUS interviews are conducted using Computer Assisted Telephone Interviewing. Procedures are in place to
collect information from the small number of households that did not provide a telephone number during the
CPS interview.

   ATUS designated people are preassigned a day of the week about which to report. Preassignment is designed to
reduce variability in response rates across the week and to allow oversampling of weekend days so that accurate
weekend day measures can be developed. Interviews occur on the day following the assigned day. For example,
a person assigned to report about a Monday would be contacted on the following Tuesday. Ten percent of
designated people are assigned to report about each of the five weekdays. Twenty-five percent are assigned to
report about each weekend day. Households are called for up to 8 consecutive weeks (for example, 8 Tuesdays)
in order to secure an interview.

About the questionnaire

   In the time diary portion of the ATUS interview, survey respondents sequentially report activities they did
between 4 a.m. on the day before the interview until 4 a.m. on the day of the interview. For each activity,
respondents are asked how long the activity lasted. For activities other than personal care activities
(such as sleeping and grooming), interviewers also ask respondents where they were and who was in the room
with them (if at home) or who accompanied them (if away from home). If respondents report doing more than
one activity at a time, they are asked to identify which one was the "main" (primary) activity. If none can
be identified, then the interviewer records the first activity mentioned. After completing the time diary,
interviewers ask respondents additional questions to clearly identify work, volunteering, eldercare, and
secondary childcare activities. Secondary childcare is defined as having a child under age 13 in one’s
care while doing other activities.

   In addition, the ATUS includes an update of the household composition from the last CPS interview (2 to 5
months prior to the ATUS interview), the labor force status of the respondent, and the employment status of
his or her spouse or unmarried partner.  For respondents who became employed or changed jobs between the
last CPS interview and the ATUS interview, information also is collected on industry, occupation, class
of worker, and earnings. Finally, a question about current school enrollment status is asked of all
respondents ages 15 to 49.

   After completing the interview, primary activity descriptions are assigned a single 6-digit code using the
ATUS Coding Lexicon. The 3-tier coding system consists of 17 major activity categories, each with multiple
second- and third-tier subcategories. These coding lexicon categories are then combined into composite
categories for publication, such as in this news release. Descriptions of categories shown in this news release
can be found in the Major activity category definitions section of this Technical Note. The ATUS Coding
Lexicon can be accessed at www.bls.gov/tus/lexicons.htm.

   Because of the complexity of coding everyday activities into narrowly defined lexicon categories, coders
use a comprehensive set of rules to guide their decisions. In order to capture useful and detailed
information, travel activities are coded according to the purpose of travel. For more information about
coding travel, see Exhibit 5.1 of the ATUS User’s Guide at www.bls.gov/tus/atususersguide.pdf.

Concepts and definitions

   Average day. The average day measure reflects an average distribution across all people in the reference
population and all days of the week. The ATUS collects data about daily activities from all segments of
the population age 15 and over, including people who are employed and not employed. Activity profiles
differ based upon age, employment status, sex, and other characteristics. For example, in 2024, people
in the United States age 15 and over spent 3.1 hours per day working. By comparison, on an average
weekday they worked, full-time employed people spent 8.4 hours working. Many activities typically are
not done on a daily basis, and some activities only are done by a subset of the population.

   Average hours per day. The average number of hours spent in a 24-hour day (between 4 a.m. on the diary day
and 4 a.m. on the interview day) doing a specified activity.

  --Average hours per day, population. The average number of hours per day is computed using all responses
    from a given sample of the population, including those of respondents who did not do a particular activity
    on their diary day. These estimates reflect how many people engaged in an activity and the amount of
    time they spent doing it.

  --Average hours per day, people who did the activity. The average number of hours per day is computed using
    only responses from those who engaged in a particular activity on their diary day.

   Diary day. The diary day is the day about which the respondent reports. For example, the diary day of a
respondent interviewed on Tuesday is the preceding Monday.

Earnings

  --Usual weekly earnings. Estimates represent the earnings of full-time wage and salary workers with one
    job only (both incorporated and unincorporated self-employed are excluded), before taxes and other
    deductions. They include any overtime pay, commissions, or tips usually received. Usual weekly earnings
    are updated in the ATUS for about 40 percent of wage and salary workers--if the respondent changed jobs
    or employment status between the CPS and ATUS interviews, or if the CPS weekly earnings value was
    imputed. This means that the earnings information could be out of date because the CPS interview was
    done 2 to 5 months prior to the ATUS interview. Respondents are asked to identify the easiest way
    for them to report earnings (hourly, weekly, biweekly, twice monthly, annually, or other) and how much
    they usually earn in the reported time period. Earnings reported on a basis other than weekly are
    converted to a weekly equivalent. The term "usual" is as perceived by the respondent. If the respondent
    asks for a definition of usual, interviewers are instructed to define the term as more than half the
    weeks worked during the past 4 or 5 months.

  --Weekly earnings quartiles. The ranges used for the quartiles represent approximately 25 percent of
    full-time wage and salary workers (both incorporated and unincorporated self-employed are excluded)
    who held only one job. For example, 25 percent of full-time wage and salary workers with one job only
    had weekly earnings of $840 or less in 2024. These dollar values vary from year to year.

Employment status

  --Employed. All people who:

    1) At any time during the 7 days prior to the interview did any work at all as paid employees, or
       worked in their own business or profession or on their own farm; or

    2) Were not working during the 7 days prior to the interview but had jobs or businesses from which
       they were temporarily absent because of illness, bad weather, vacation, childcare problems,
       labor-management disputes, maternity or paternity leave, job training, or other family or personal
       reasons, whether or not they were paid for the time off or were seeking other jobs; or

    3) Usually worked 15 hours or more as unpaid workers in a family-operated enterprise.

  --Employed full time. Full-time workers are those who usually worked 35 or more hours per week at
    all jobs combined.

  --Employed part time. Part-time workers are those who usually worked fewer than 35 hours per week at
    all jobs combined.

  --Not employed. People are not employed if they do not meet the conditions for employment. Those
    who are not employed include individuals classified as unemployed as well as those classified as not
    in the labor force (using CPS definitions).

   The numbers of employed and not employed people in this release do not correspond to published totals
from the CPS for several reasons. First, the reference population for the ATUS is age 15 and over, whereas
it is age 16 and over for the CPS. Second, ATUS data are collected continuously, the employment reference
period being the 7 days prior to the interview. By contrast, CPS data are usually collected during the
week including the 19th of the month and generally refer to employment during the week containing the
12th of the month. Finally, the CPS accepts answers from household members about other household members
whereas such proxy responses are not allowed in the ATUS. While the information on employment from the
ATUS is useful for assessing work in the context of other daily activities, the employment data are not
intended for analysis of current employment trends. Compared with the CPS and other estimates of
employment, the ATUS estimates are based on a much smaller sample and are only available with a
substantial lag since ATUS data and estimates are published during the year following data collection.

   Household children. Household children are children under age 18 residing in the household of the
ATUS respondent. The children may be related to the respondent (such as his or her own children,
grandchildren, nieces or nephews, or brothers or sisters) or not related (such as foster children
or children of roommates or boarders).

   Primary activity. A primary activity is the main activity a respondent was doing at a specified time.
With the exception of secondary childcare in table 10, the estimates presented in this release reflect
time spent in primary activities only.

   Secondary activities. A secondary (or simultaneous) activity is an activity done at the same time as
a primary activity. With the exception of the care of children under age 13, information on
secondary activities is not collected in the ATUS.

   Secondary childcare. Secondary childcare is care for children under age 13 that is done while doing
an activity other than primary childcare, such as cooking dinner. Secondary childcare estimates are
derived by summing the durations of activities during which respondents had at least one child under
age 13 in their care while doing other things. The time individuals spend providing secondary childcare
is further restricted to the time between when the first household child under age 13 woke up and when
the last household child under age 13 went to bed. It is also restricted to times the respondent was
awake. If respondents report providing both primary and secondary care at the same time, the time is
attributed to primary care only.

   Weekday, weekend, and holiday estimates. Estimates for weekdays are an average of reports about Monday
through Friday, excluding holidays. Estimates for weekend days and holidays are an average of reports
about Saturdays, Sundays, and the following holidays: New Year’s Day, Easter, Memorial Day, the Fourth
of July, Labor Day, Thanksgiving Day, and Christmas Day.

Major activity category definitions

   The following definitions describe the activity categories shown in this report. All major time-use
categories in this release include related travel time and waiting time. For example, time spent
"driving to the stadium" and time spent "waiting to get into the stadium to play ball" are included
in Leisure and sports.

   Personal care activities. Personal care activities include sleeping, grooming (such as bathing or dressing),
health-related self-care, and personal or private activities. Receiving unpaid personal care from
others (for example, "my sister put polish on my nails") also is captured in this category. In general,
respondents are not asked who they were with or where they were for personal care activities, as such
information can be sensitive.

   Eating and drinking. All time spent eating or drinking (except eating and drinking done as part of a work
or volunteer activity), whether alone, with others, at home, at a place of purchase, or somewhere else,
is classified here.

   Household activities. Household activities are activities done by individuals to maintain their households.
These include housework; cooking; lawn and garden care; pet care; vehicle maintenance and repair; home
maintenance, repair, decoration, and renovation; and household management and organizational activities
(such as filling out paperwork or planning a party). Food preparation, whether or not reported as done
specifically for another household member, is always classified as a household activity unless it was
done as a volunteer, work, or income-generating activity. For example, "making breakfast for my son" is
coded as a household activity, not as childcare.

   Purchasing goods and services. This category includes time spent purchasing consumer goods, professional
and personal care services, household services, and government services. Consumer purchases include most
purchases and rentals of consumer goods, regardless of the mode or place of purchase or rental (in person,
online, via telephone, at home, or in a store). Gasoline, grocery, other food purchases, and all other
shopping are further broken out in subcategories.

   Time spent obtaining, receiving, and purchasing professional and personal care services provided by
someone else also is classified in this category. Professional services include childcare, financial
services and banking, legal services, medical and adult care services, real estate services, and veterinary
services. Personal care services include day spas, hair salons and barbershops, nail salons, and tanning
salons. Activities classified here include time spent paying, meeting with, or talking to service
providers, as well as time spent receiving the service or waiting to receive the service.

   Time spent arranging for and purchasing household services provided by someone else also is classified here.
Household services include housecleaning; cooking; lawn care and landscaping; pet care; tailoring,
laundering, and dry cleaning; vehicle maintenance and repairs; and home repairs, maintenance, and
construction.

   This category also captures the time spent obtaining government services--such as applying for food
assistance and purchasing government-required licenses or paying fines or fees.

   Caring for and helping household members. Time spent doing activities to care for or help any child
(under age 18) or adult in the household, regardless of relationship to the respondent or the physical
or mental health status of the person being helped, is classified here. Caring for and helping activities
for household children and adults are coded separately in subcategories.

   Primary childcare activities include time spent providing physical care; playing with children; reading
with children; assisting with homework; attending children's events; taking care of children's health
needs; and dropping off, picking up, and waiting for children. Passive childcare done as a primary activity
(such as "keeping an eye on my son while he swam in the pool") also is included. A child's presence during
the activity is not enough in itself to classify the activity as childcare. For example, "watching
television with my child" is coded as a leisure activity, not as childcare.

   Secondary childcare occurs when people have a child under age 13 "in their care" while doing activities
other than primary childcare. For a complete definition, see the Concepts and definitions section of
this Technical Note.

   Caring for and helping household members also includes a range of activities done to benefit adult members
of households, such as providing physical and medical care or obtaining medical services. Doing something
as a favor for or helping another household adult does not automatically result in classification as a
helping activity. For example, a report of "helping my spouse cook dinner" is considered a household
activity (food preparation), not a helping activity, because cooking dinner benefits the household as
a whole. By contrast, doing paperwork for another person usually benefits the individual, so a report
of "filling out an insurance application for my spouse" is considered a helping activity.

   Caring for and helping nonhousehold members. This category includes time spent in activities done to care
for or help others--both children (under age 18) and adults--who do not live in the household. When done
for or through an organization, time spent helping nonhousehold members is classified as volunteering,
rather than as helping nonhousehold members. Care of nonhousehold children, even when done as a favor or
helping activity for another adult, is always classified as caring for and helping nonhousehold children,
not as helping another adult.

   Working and work-related activities. This category includes time spent working, doing activities as part
of one's job, engaging in income-generating activities not as part of one's job, and job search activities.
"Working" includes hours spent doing the specific tasks required of one's main or other job, regardless of
location or time of day. "Work-related activities" include activities that are not obviously work but are
done as part of one's job, such as having a business lunch and playing golf with clients. "Other income
-generating activities" are those done "on the side" or under informal arrangement and are not part of a
regular job. Such activities might include selling homemade crafts, maintaining a rental property, or having
a yard sale. These activities are those for which individuals are paid or will be paid.

   Travel time related to working and work-related activities includes time spent traveling to and from work,
as well as time spent traveling for work-related, income-generating, and job search activities.

   Educational activities. Time spent taking classes for a degree or for personal interest (including
attending school virtually and taking internet or other distance-learning courses), time spent doing
research and homework, and time spent taking care of administrative tasks related to education (such as
registering for classes or obtaining a school ID) are included in this category. For high school students,
before- and after-school extracurricular activities (except sports) also are classified as educational
activities. Educational activities do not include time spent for classes or training received as part
of a job. Time spent helping others with their education-related activities is classified as an activity
involving caring for and helping others.

   Organizational, civic, and religious activities. This category captures time spent volunteering for or
through an organization, performing civic obligations, and participating in religious and spiritual
activities. Civic obligations include government-required duties, such as serving jury duty or appearing
in court, and activities that assist or influence government processes, such as voting or attending
town hall meetings. Religious activities include those normally associated with membership in or
identification with specific religions or denominations, such as attending religious services; participating
in choirs, youth groups, or unpaid teaching (unless identified as volunteer activities); and engaging in
personal religious practices, such as praying.

   Leisure and sports. The leisure and sports category includes time spent in sports, exercise, and
recreation; socializing and communicating; and other leisure activities. Sports, exercise, and recreation
activities include participating in--as well as attending or watching--sports, exercise, and recreational
activities. Recreational activities include yard games like croquet or horseshoes, as well as activities
like billiards and dancing. Socializing and communicating includes face-to-face social communication and
hosting or attending social functions. Leisure activities include watching television; reading; relaxing
or thinking; playing computer, board, or card games; using a computer or the internet for personal
interest; playing or listening to music; and other activities, such as attending arts, cultural, and
entertainment events.

   Telephone calls, mail, and e-mail. This category captures time spent in telephone communication and
household or personal mail or e-mail. This category also includes texting and internet voice and video
calling. Telephone and internet purchases are classified in Purchasing goods and services. Telephone calls,
mail, or e-mail identified as related to work or volunteering are classified as work or volunteering.

   Other activities, not elsewhere classified. This residual category includes security procedures related
to traveling, traveling not associated with a specific activity category, ambiguous activities that
could not be coded, and missing activities. Missing activities result when respondents did not remember
what they did for a period of time, or when they considered an activity too private or personal to report.

Processing and estimation

   After ATUS data are collected, they go through an editing and imputation procedure. Responses to CPS
questions that are re-asked in the ATUS go through the regular CPS edit and imputation procedures. Some
item nonresponses for questions unique to the ATUS (such as where an activity took place or how much
time was spent doing secondary childcare) also are imputed. Missing activities and missing values for
who was present during an activity are never imputed.

   ATUS records are weighted quarterly to reduce bias in the estimates due to differences in sampling and
response rates across subpopulations and days of the week. Specifically, the data are weighted to ensure
the following:

  --Weekdays represent about 5/7 of the weighted data, and weekend days represent about 2/7 of the
    weighted data for the population as a whole and for selected subpopulations. The actual proportions
    depend on the number of weekdays and weekend days in a given quarter.

  --The sum of the weights is equal to the number of person-days in the quarter for the population as
    a whole and for selected subpopulations.

Reliability of the estimates

   Statistics based on the ATUS are subject to both sampling and nonsampling error. When a sample, rather than
the entire population, is surveyed, estimates differ from the true population values they represent. The
component of this difference that occurs because samples differ by chance is known as sampling error,
and its variability is measured by the standard error of the estimate.

   Sample estimates from a given survey design are unbiased when an average of the estimates from all
possible samples would yield, hypothetically, the true population value. In this case, the sample estimate
and its standard error can be used to construct approximate confidence intervals, or ranges of values that
include the true population value with known probabilities. If the process of selecting a sample from the
population were repeated many times, an estimate made from each sample, and a suitable estimate of its
standard error calculated for each sample, then approximately 90 percent of the intervals from 1.645
standard errors below the estimate to 1.645 standard errors above the estimate would include the true
population value. BLS analyses are generally conducted at the 90-percent level
of confidence.

   The ATUS data also are affected by nonsampling error, which is the average difference between population
and sample values for samples generated by a given process. Nonsampling error can occur for many reasons,
including failure to sample a segment of the population, inability to obtain information for all people in
the sample, inability or unwillingness of respondents to provide correct information, and errors made in
the collection or processing of the data. Errors also could occur if nonresponse is correlated with time
use.

Publication requirements

   Estimates of average hours per day and participation rates are not published unless there are a minimum
number of respondents representing the given population. Additional publication criteria are applied
that include the number of respondents who reported doing a specified activity and the standard error or
coefficient of variation for the estimate. Estimates that are considered "close to zero" or that round
to 0.00, are published as approximately zero or "z". For a detailed description of the statistical reliability
criteria necessary for publication, please contact ATUS staff at ATUSinfo@bls.gov.
"""

In [ ]:
prompt = """
Following is technical notes of a survey that you will need to replicate.
The technical notes does not have the results, but only the methodology, which you will need to follow.

## TECHNICAL NOTES ##
{technical_notes}

## TASK ##
Your task is to estimate the average daily hours spent, for each of the following activities:
{activities}

Respond only with a json object (and nothing else), where each key is the activity, and the value is the estimated average daily hours spent (number between 0 to 24)
"""

In [ ]:
gemini_api_key = userdata.get('gemini_api_key')
client = genai.Client(api_key=gemini_api_key)

model_name = 'gemini-2.5-flash-lite'
response = client.models.generate_content(
    model=model_name,
    contents=prompt.format(
        technical_notes=technical_notes,
        activities='\n'.join(df_gt['activity'].tolist()),
    ),
    config=types.GenerateContentConfig(
        temperature=0
    )
)
print(response.text)

```json
{
  "Personal care, including sleep": 9.0,
  "Eating and drinking": 1.5,
  "Household activities": 2.0,
  "Purchasing goods and services": 0.5,
  "Caring for and helping household members": 1.0,
  "Caring for and helping nonhousehold members": 0.2,
  "Working and work-related activities": 3.0,
  "Educational activities": 1.0,
  "Organizational, civic, and religious activities": 0.3,
  "Leisure and sports": 4.0,
  "Telephone calls, mail, and e-mail": 0.5,
  "Other activities, not elsewhere classified": 0.1
}
```


In [ ]:
print(prompt.format(
    technical_notes=technical_notes,
    activities='\n'.join(df_gt['activity'].tolist())
))


Following is technical notes of a survey that you will need to replicate.
The technical notes does not have the results, but only the methodology, which you will need to follow.

## TECHNICAL NOTES ##

Technical Note
 

   The estimates in this news release are based on annual average data from the American Time Use Survey (ATUS).
The ATUS, which is conducted by the U.S. Census Bureau for the Bureau of Labor Statistics (BLS), is a
continuous survey about how individuals age 15 and over spend their time.

   If you are deaf, hard of hearing, or have a speech disability, please dial 7-1-1 to access telecommunications
relay services. 

Survey methodology

   Data collection for the ATUS began in January 2003. Sample cases for the survey are selected monthly, and 
interviews are conducted continuously throughout the year. In 2024, approximately 7,700 individuals were
interviewed. Estimates are released annually.

   ATUS sample households are chosen from the households that completed thei

In [ ]:
df_pred = (
    pd.Series(json.loads(response.text.removeprefix('```json').removesuffix('```')))
    .reset_index()
    .rename(columns={'index': 'activity', 0: 'hours_spent_pred'})
)

In [ ]:
df_pred

,activity,hours_spent_pred
0,"Personal care, including sleep",9.0
1,Eating and drinking,1.5
2,Household activities,2.0
3,Purchasing goods and services,0.5
4,Caring for and helping household members,1.0
5,Caring for and helping nonhousehold members,0.2
6,Working and work-related activities,3.0
7,Educational activities,1.0
8,"Organizational, civic, and religious activities",0.3
9,Leisure and sports,4.0


In [ ]:
df_merged = df_pred.merge(df_gt, on='activity')
df_merged

,activity,hours_spent_pred,hours_spent_gt
0,"Personal care, including sleep",9.0,9.80
1,Eating and drinking,1.5,1.24
2,Household activities,2.0,2.01
3,Purchasing goods and services,0.5,0.67
4,Caring for and helping household members,1.0,0.51
5,Caring for and helping nonhousehold members,0.2,0.17
6,Working and work-related activities,3.0,3.43
7,Educational activities,1.0,0.42
8,"Organizational, civic, and religious activities",0.3,0.30
9,Leisure and sports,4.0,5.07


In [ ]:
mean_absolute_percentage_error(
    y_true=df_merged['hours_spent_gt'],
    y_pred=df_merged['hours_spent_pred']
)

0.47551841007971846

# Add a new dimention - age groups

In [ ]:
age_groups = [
    f'{age} to {age + 10} years'
    for age in range(15, 100, 10)
]

age_groups

['15 to 25 years',
 '25 to 35 years',
 '35 to 45 years',
 '45 to 55 years',
 '55 to 65 years',
 '65 to 75 years',
 '75 to 85 years',
 '85 to 95 years',
 '95 to 105 years']

In [ ]:
prompt = """
Following is technical notes of a survey that you will need to replicate.
The technical notes does not have the results, but only the methodology, which you will need to follow.

## TECHNICAL NOTES ##
{technical_notes}

## TASK ##
Your task is to estimate the average daily hours spent, for each of the following activities and age groups:

age groups:
{age_groups}

activities:
{activities}


You must respond with your estimation of the survey results, even though you don't have access to the data.
Please respond only with a json array where each element has the following keys:
- activity
- age_group
- hours_spent: average daily hours spent (number between 0 to 24)
"""

In [ ]:
response = client.models.generate_content(
    model=model_name,
    contents=prompt.format(
        technical_notes=technical_notes,
        activities='\n'.join(df_gt['activity'].tolist()),
        age_groups = '\n'.join(age_groups),
    ),
    config=types.GenerateContentConfig(
        temperature=0
    )
)
print(response.text)

```json
[
  {
    "activity": "Personal care, including sleep",
    "age_group": "15 to 25 years",
    "hours_spent": 9.5
  },
  {
    "activity": "Eating and drinking",
    "age_group": "15 to 25 years",
    "hours_spent": 1.5
  },
  {
    "activity": "Household activities",
    "age_group": "15 to 25 years",
    "hours_spent": 1.0
  },
  {
    "activity": "Purchasing goods and services",
    "age_group": "15 to 25 years",
    "hours_spent": 0.8
  },
  {
    "activity": "Caring for and helping household members",
    "age_group": "15 to 25 years",
    "hours_spent": 0.5
  },
  {
    "activity": "Caring for and helping nonhousehold members",
    "age_group": "15 to 25 years",
    "hours_spent": 0.2
  },
  {
    "activity": "Working and work-related activities",
    "age_group": "15 to 25 years",
    "hours_spent": 3.0
  },
  {
    "activity": "Educational activities",
    "age_group": "15 to 25 years",
    "hours_spent": 3.5
  },
  {
    "activity": "Organizational, civic, and religiou

In [ ]:
df_age_pred = (
    pd.DataFrame(json.loads(response.text.removeprefix('```json').removesuffix('```')))
    .rename(columns={'hours_spent': 'hours_spent_pred'})
)

df_age_pred = df_age_pred.groupby(['activity'], as_index=False)['hours_spent_pred'].mean()
df_age_pred

,activity,hours_spent_pred
0,Caring for and helping household members,1.011111
1,Caring for and helping nonhousehold members,0.244444
2,Eating and drinking,1.500000
3,Educational activities,0.555556
4,Household activities,1.466667
5,Leisure and sports,2.855556
6,"Organizational, civic, and religious activities",0.444444
7,"Other activities, not elsewhere classified",0.188889
8,"Personal care, including sleep",9.111111
9,Purchasing goods and services,0.711111


In [ ]:
df_gt['hours_spent_gt'].sum()

np.float64(24.01)

In [ ]:
df_pred['hours_spent_pred'].sum()

np.float64(23.1)

In [ ]:
df_age_pred['hours_spent_pred'].sum()

np.float64(22.366666666666667)

In [ ]:
df_age_merged = df_age_pred.merge(df_gt, on='activity')
df_age_merged

,activity,hours_spent_pred,hours_spent_gt
0,Caring for and helping household members,1.011111,0.51
1,Caring for and helping nonhousehold members,0.244444,0.17
2,Eating and drinking,1.500000,1.24
3,Educational activities,0.555556,0.42
4,Household activities,1.466667,2.01
5,Leisure and sports,2.855556,5.07
6,"Organizational, civic, and religious activities",0.444444,0.30
7,"Other activities, not elsewhere classified",0.188889,0.21
8,"Personal care, including sleep",9.111111,9.80
9,Purchasing goods and services,0.711111,0.67


In [ ]:
mean_absolute_percentage_error(
    y_true=df_age_merged['hours_spent_gt'],
    y_pred =df_age_merged['hours_spent_pred']
)

0.3841211744495505